## Phase 2, Step 1: which ownership links the 50% calculation will use

**The goal of Phase 2:** under the US "50% rule", a company is treated as sanctioned if sanctioned owners together hold 50% or more of it, directly or through other companies they control. To calculate that, we first need a clean list of who owns whom.

### What one row means
Each row in silver.ownership is one ownership link: **one owner holds a share of one company.**
Example: "Company A owns 60% of Company B."

| Column | Plain meaning | Example |
|---|---|---|
| ownership_id | the ID of this ownership link | ru-6096a14f... |
| owner_ref | who owns, exactly as the source wrote it | ru-inn-7707329152 |
| asset_ref | what is owned, exactly as the source wrote it | ru-inn-4011017003 |
| owner_id | who owns, using our standard ID | ru-inn-7707329152 |
| asset_id | what is owned, using our standard ID | ru-inn-4011017003 |

**Standard ID (canonical_id):** the same company often appears in several sources under different IDs. OpenSanctions recognises these as one company and gives them one shared ID. Example: Mekhatronika appears as ru-inn-3702686347 (Russian register) and permid-5086648493 (another source), and both map to NK-R5MBqWzCxd5iUvZzYTctC3. We use this shared ID everywhere, so each company appears once.

- Several owners of one company, or one owner of several companies, show up as several rows.
- owner_id = asset_id would mean a company owns itself. 247 such rows exist; they are data errors and are left out.

### What we checked
- Translating to the standard ID changed nothing: on all 206,390 links, the source already used the standard ID. We keep the step anyway, in case future data doesn't.
- Every company that owns or is owned has a matching row in silver.legal_entity (0 missing). The only exceptions are things that aren't companies: people, vessels, aircraft and other assets.

### Profiling the current links (Nov 1)
| Measure | Result |
|---|---|
| Current links (excluding companies owning themselves) | 110,079 |
| Distinct owner -> company pairs | 109,513 |
| Pairs reported more than once | 509 (0.5%) |
| Pairs with no ownership percentage | 38,287 (35%) |

When a pair is reported more than once, it is one of two things:
- **The same stake reported twice**, usually by two different sources (e.g. 72.2% and 72.24%). Adding them would wrongly double it.
- **Two separate stakes** held by the same owner (e.g. 49% and 51%), which together really are 100%.

### Decisions
| Decision | Why |
|---|---|
| Use current links only | an ownership that has ended can't make a company sanctioned today |
| Leave out companies owning themselves | they would count their own shares |
| One row per owner -> company pair, taking the larger percentage | never double-counts; the rare "two separate stakes" case is slightly undercounted (51% instead of 100%), which rarely changes a 50% result |
| Keep links with no percentage | they can't count towards 50%, but they still show who is connected to whom |
| Keep a low and a high percentage | some sources give ranges (e.g. 75-100%): the low value tells us "sanctioned for sure", the high value "possibly sanctioned" |

In [0]:
WITH pairs AS (
    SELECT owner_id, asset_id,
           count(*) AS n_records,
           min(pct_min) AS pct_lo,
           max(pct_max) AS pct_hi,
           count(pct_min) AS n_with_pct
    FROM pep_dev.silver.ownership
    WHERE is_current AND NOT is_self_owning
    GROUP BY owner_id, asset_id
)

SELECT count(*) as n_pairs, 
count_if(n_records>1) as n_multi,
count_if(n_with_pct=0)  as n_no_pct,
sum(n_records) as n_edges
FROM pairs

In [0]:
  SELECT owner_id, asset_id,
           count(*) AS n_records,
           min(pct_min) AS pct_lo,
           max(pct_max) AS pct_hi,
           count(pct_min) AS n_with_pct
    FROM pep_dev.silver.ownership
    WHERE is_current AND NOT is_self_owning 
    GROUP BY owner_id, asset_id
    HAVING n_records>1;

    SELECT * FROM pep_dev.silver.ownership 
    WHERE owner_id='NK-KgKrVitX9Pv7S4TNvQbZCA' AND asset_id='NK-j2qrz9qzrK2dPNaQkEBZFU'

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.edges_current AS 
SELECT owner_id,
       asset_id,
       owner_type,
       asset_type,
       count(*) AS n_records,
       max(pct_min) AS pct_lo,
       max(pct_max) AS pct_hi
FROM pep_dev.silver.ownership
WHERE is_current
    AND NOT is_self_owning
GROUP BY owner_id,asset_id,owner_type,asset_type;

SELECT * FROM pep_dev.eval.edges_current;


In [0]:
SELECT count(*), count(pct_lo) FROM pep_dev.eval.edges_current

In [0]:
DESCRIBE pep_dev.silver.person

In [0]:
SELECT count(DISTINCT e.owner_id)                         AS person_owners,
       count(DISTINCT CASE WHEN p.person_id IS NOT NULL
                           THEN e.owner_id END)           AS found_in_person
FROM pep_dev.eval.edges_current AS e
LEFT JOIN pep_dev.silver.person AS p
  ON p.person_id = e.owner_id
WHERE e.owner_type = 'Person'

In [0]:
SELECT count_if(is_sanctioned)                                         AS flag_true,
       count_if(array_contains(topics, 'sanction'))                    AS exact_sanction,
       count_if(is_sanctioned AND NOT array_contains(topics, 'sanction')) AS mismatch
FROM pep_dev.silver.person  

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.seeds AS 
SELECT canonical_id AS node_id, entity_type AS node_type
FROM pep_dev.silver.legal_entity
WHERE is_sanctioned
UNION ALL
SELECT person_id AS node_id, 'Person' AS node_type
FROM pep_dev.silver.person
WHERE is_sanctioned


In [0]:
SELECT count(*) FROM pep_dev.eval.seeds


In [0]:
SELECT count(*), entity_type FROM pep_dev.silver.legal_entity
group by entity_type

In [0]:
-- 1. one row per node, no overlap between companies and people
SELECT count(*), count(DISTINCT node_id) FROM pep_dev.eval.seeds;



In [0]:
-- 2. how many seeds actually own something today (where the cascade starts)
SELECT s.node_type, count(DISTINCT s.node_id) AS seeds_that_own
FROM pep_dev.eval.seeds AS s
JOIN pep_dev.eval.edges_current AS e ON e.owner_id = s.node_id
GROUP BY s.node_type;

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.blocked AS   
SELECT node_id,node_type, 0 AS round  
FROM pep_dev.eval.seeds

In [0]:
--ROUND 1 BY HAND

INSERT INTO pep_dev.eval.blocked
SELECT e.asset_id, e.asset_type, 1 
FROM
pep_dev.eval.edges_current AS e
JOIN pep_dev.eval.blocked AS b
ON e.owner_id = b.node_id
WHERE 
e.asset_id NOT IN (SELECT node_id FROM pep_dev.eval.blocked)
GROUP BY e.asset_id, e.asset_type
HAVING sum(e.pct_lo) >= 50

In [0]:
-- %python
-- # ------------------------------------------------------------------
-- # OFAC 50% cascade, as a "repeat until nothing changes" loop
-- #
-- # Rule: a company is blocked if blocked owners TOGETHER hold 50% or
-- #       more of it. Once blocked, it counts as a blocked owner for the
-- #       companies it owns.
-- #
-- # round 0 = seeds (already on a sanctions list)
-- # round N = companies pushed to 50%+ by owners blocked in earlier rounds
-- # Each round reaches one layer further down the ownership chain.
-- #
-- # Always stops: the blocked list only grows and a blocked company is
-- # never added again, so circular ownership can't loop forever.
-- # Each round is written to Delta, so Spark's plan doesn't keep growing.
-- # ------------------------------------------------------------------

-- def run_cascade(target_table: str, pct_col: str, max_rounds: int = 20) -> None:
--     """pct_col = 'pct_lo' -> blocked for sure, 'pct_hi' -> possibly blocked."""

--     # Round 0: start (or restart) the blocked list from the seeds
--     spark.sql(f"""
--         CREATE OR REPLACE TABLE {target_table} AS
--         SELECT node_id, node_type, 0 AS round
--         FROM pep_dev.eval.seeds
--     """)

--     for r in range(1, max_rounds + 1):
--         res = spark.sql(f"""
--             INSERT INTO {target_table}
--             SELECT e.asset_id, e.asset_type, :r            -- tag new rows with this round
--             FROM pep_dev.eval.edges_current AS e
--             JOIN {target_table} AS b
--               ON e.owner_id = b.node_id                    -- only links whose OWNER is blocked
--             WHERE e.asset_id NOT IN (SELECT node_id FROM {target_table})  -- skip already blocked
--             GROUP BY e.asset_id, e.asset_type              -- one bucket per owned company
--             HAVING sum(e.{pct_col}) >= 50                  -- blocked owners TOGETHER hold 50%+
--         """, args={"r": r})

--         n = res.collect()[0]["num_inserted_rows"]          # companies blocked this round
--         print(f"{target_table} round {r}: {n} newly blocked")
--         if n == 0:                                         # nothing new: cascade complete
--             break

In [0]:
%python
# ------------------------------------------------------------------
# run_cascade: the "repeat until nothing changes" loop
#
# round 0 = seeds (already on a sanctions list)
# round N = companies added because of owners added in earlier rounds
# Each round reaches one layer further down the ownership chain.
# Always stops: the list only grows and nothing is added twice.
# ------------------------------------------------------------------
def run_cascade(target_table: str,                                # where to save the result
                pct_col: str,                                     # "pct_lo" (low end) or "pct_hi" (high end)
                min_pct: float = 50,                              # 50 = OFAC rule, 0 = any link counts
                edges_table: str = "pep_dev.eval.edges_current",  # current links, or all links
                max_rounds: int = 20) -> None:                    # safety cap

    # Round 0: start the list from the seeds
    spark.sql(f"""
        CREATE OR REPLACE TABLE {target_table} AS
        SELECT node_id, node_type, 0 AS round
        FROM pep_dev.eval.seeds
    """)

    for r in range(1, max_rounds + 1):
        res = spark.sql(f"""
            INSERT INTO {target_table}
            SELECT e.asset_id, e.asset_type, :r                -- the owned company + this round's number
            FROM {edges_table} AS e
            JOIN {target_table} AS b
              ON e.owner_id = b.node_id                        -- only links whose OWNER is on the list
            WHERE e.asset_id NOT IN (SELECT node_id FROM {target_table})  -- skip companies already on it
            GROUP BY e.asset_id, e.asset_type                  -- one bucket per owned company
            HAVING sum(coalesce(e.{pct_col}, 0)) >= {min_pct}  -- owners on the list together hold enough
        """, args={"r": r})

        n = res.collect()[0]["num_inserted_rows"]              # how many companies this round added
        print(f"{target_table} round {r}: {n} added")
        if n == 0:                                             # nothing new: the list is complete
            break

### Edge table with ended links included
Same as `edges_current`, minus the `is_current` filter. Used to test whether a company is connected only through ownership that has ended.

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.edges_all AS
SELECT owner_id,                    -- who owns (group ID)
       asset_id,                    -- what is owned (group ID)
       owner_type,
       asset_type,
       count(*)     AS n_records,   -- records squashed into this pair
       max(pct_min) AS pct_lo,      -- larger report, never the sum
       max(pct_max) AS pct_hi
FROM pep_dev.silver.ownership
WHERE NOT is_self_owning            -- no is_current filter: ended links stay
GROUP BY owner_id, asset_id, owner_type, asset_type

In [0]:
%python
run_cascade("pep_dev.eval.blocked",     "pct_lo")              # blocked for sure          (expect 11,091 over 6 rounds)
run_cascade("pep_dev.eval.blocked_hi",  "pct_hi")              # possibly blocked          (expect 11,102)
run_cascade("pep_dev.eval.reached",     "pct_lo", min_pct=0)   # connected today           (expect 31,062 over 10 rounds)
run_cascade("pep_dev.eval.reached_all", "pct_lo", min_pct=0,
            edges_table="pep_dev.eval.edges_all")              # connected ever, ended too (expect 61,356)

### Comparing with OpenSanctions' sanction.control

In [0]:
-- expect: 11,091 | 11,091 | 0  → our cascade is correct (blocked ⊂ their flag)
SELECT count(*)                                                    AS we_blocked,
       count_if(array_contains(le.topics, 'sanction.control'))     AS they_also_flag,
       count_if(NOT array_contains(le.topics, 'sanction.control')) AS only_we_flag
FROM pep_dev.eval.blocked AS b
JOIN pep_dev.silver.legal_entity AS le ON le.canonical_id = b.node_id
WHERE b.round > 0                                  -- skip the listed parties themselves

In [0]:
%python
from datetime import date

stale = {row.id for row in spark.table("pep_dev.eval.stale_67").collect()}   # the 67

def flagged_below(start_nodes):
    """Which of the 67 sit at or below these nodes (following current links down the chains)?"""
    below, frontier = set(start_nodes), set(start_nodes)
    while frontier:
        spark.createDataFrame([(x,) for x in frontier], ["id"]).createOrReplaceTempView("frontier")
        kids = {row.asset_id for row in spark.sql("""
            SELECT e.asset_id
            FROM pep_dev.eval.edges_current AS e
            JOIN frontier              AS f ON f.id = e.owner_id   -- links OUT of the current layer
            JOIN pep_dev.eval.upstream AS u ON u.id = e.asset_id   -- stay inside the disconnected chains
        """).collect()}
        frontier = kids - below
        below |= frontier
    return below & stale

# The 45: flagged companies below breaks from 2022-2025
breaks = spark.sql("""
    SELECT o.asset_id, o.end_date
    FROM pep_dev.silver.ownership AS o
    JOIN pep_dev.eval.upstream AS u ON u.id = o.asset_id
    JOIN pep_dev.eval.reached  AS r ON r.node_id = o.owner_id
    WHERE NOT o.is_current AND NOT o.is_self_owning
""").collect()
old = flagged_below({b.asset_id for b in breaks if b.end_date < date(2026, 1, 1)})
print("the 45 (old breaks):", len(old))                          # expect 45

# Mutual pairs (A owns B and B owns A, both current) inside the disconnected chains
cycle_nodes = {row.id for row in spark.sql("""
    SELECT DISTINCT a.owner_id AS id
    FROM pep_dev.eval.edges_current AS a
    JOIN pep_dev.eval.edges_current AS b
      ON b.owner_id = a.asset_id AND b.asset_id = a.owner_id   -- the link back
    JOIN pep_dev.eval.upstream AS u1 ON u1.id = a.owner_id
    JOIN pep_dev.eval.upstream AS u2 ON u2.id = a.asset_id
    WHERE a.owner_id <> a.asset_id
""").collect()}
print("companies in mutual loops:", len(cycle_nodes))

in_or_below = flagged_below(cycle_nodes)
print("of the 67, in or below a loop:", len(in_or_below))
print("of the 45, in or below a loop:", len(in_or_below & old))

In [0]:
-- expect: 11,091 | 11 | 18,052
SELECT count_if(lo.node_id IS NOT NULL)                       AS blocked_for_sure,
       count_if(lo.node_id IS NULL AND hi.node_id IS NOT NULL) AS possibly_blocked,
       count_if(lo.node_id IS NULL AND hi.node_id IS NULL)     AS connected_not_blocked
FROM pep_dev.silver.legal_entity AS le
LEFT JOIN (SELECT node_id FROM pep_dev.eval.blocked    WHERE round > 0) AS lo ON lo.node_id = le.canonical_id
LEFT JOIN (SELECT node_id FROM pep_dev.eval.blocked_hi WHERE round > 0) AS hi ON hi.node_id = le.canonical_id
WHERE array_contains(le.topics, 'sanction.control')   -- companies they flag
  AND NOT le.is_sanctioned                            -- that aren't listed themselves

In [0]:
-- expect: 11,091 | 11 | 18,052
SELECT count_if(lo.node_id IS NOT NULL)                       AS blocked_for_sure,
       count_if(lo.node_id IS NULL AND hi.node_id IS NOT NULL) AS possibly_blocked,
       count_if(lo.node_id IS NULL AND hi.node_id IS NULL)     AS connected_not_blocked
FROM pep_dev.silver.legal_entity AS le
LEFT JOIN (SELECT node_id FROM pep_dev.eval.blocked    WHERE round > 0) AS lo ON lo.node_id = le.canonical_id
LEFT JOIN (SELECT node_id FROM pep_dev.eval.blocked_hi WHERE round > 0) AS hi ON hi.node_id = le.canonical_id
WHERE array_contains(le.topics, 'sanction.control')   -- companies they flag
  AND NOT le.is_sanctioned                            -- that aren't listed themselves

In [0]:
-- expect: 29,083 | 13 | 71   (99.8% of their flags reproduced)
SELECT count_if(r.node_id IS NOT NULL AND array_contains(le.topics, 'sanction.control'))     AS both,
       count_if(r.node_id IS NOT NULL AND NOT array_contains(le.topics, 'sanction.control')) AS we_reach_they_dont_flag,
       count_if(r.node_id IS NULL     AND array_contains(le.topics, 'sanction.control'))     AS they_flag_we_cant_reach
FROM pep_dev.silver.legal_entity AS le
LEFT JOIN (SELECT node_id FROM pep_dev.eval.reached WHERE round > 0) AS r ON r.node_id = le.canonical_id
WHERE NOT le.is_sanctioned                         -- no filter on their flag: we need both directions

### The 13 we reach but they don't flag
Finding: all 13 are mining sites from one source (`c4ads_xinjiang_mining`), owned by a listed company or one reached from it. Their flag passes through none of that source's links (0 of 13). A question for them, not an error. These links have no percentages, so they never affect the 50% result.

In [0]:
WITH gap AS (                                                    -- the 13
    SELECT le.canonical_id, le.name
    FROM pep_dev.silver.legal_entity AS le
    JOIN pep_dev.eval.reached AS r ON r.node_id = le.canonical_id AND r.round > 0   -- we reach it
    WHERE NOT le.is_sanctioned
      AND NOT array_contains(le.topics, 'sanction.control')                       -- they don't flag it
)
SELECT g.canonical_id, g.name,
       o.owner_id, o.owner_type, ro.round AS owner_round,       -- the owner that reaches it
       o.pct_min, o.start_date, o.end_date, o.datasets          -- the link and its source
FROM gap AS g
JOIN pep_dev.silver.ownership AS o
  ON o.asset_id = g.canonical_id AND o.is_current AND NOT o.is_self_owning
JOIN pep_dev.eval.reached AS ro ON ro.node_id = o.owner_id
ORDER BY g.canonical_id

In [0]:
-- expect: 6,989 | 544 | 6,989  → every record has a readable end date; 544 end after the snapshot (permit expiry)
SELECT count(*)             AS records,
       count_if(is_current) AS counted_current,
       count(end_date)      AS has_end_date            -- count(column) skips NULLs
FROM pep_dev.silver.ownership
WHERE array_contains(datasets, 'c4ads_xinjiang_mining')

In [0]:
-- expect: 13 | 0  → their flag doesn't pass through this source at all
SELECT count(DISTINCT o.asset_id)                                  AS assets_reached_via_c4ads,
       count(DISTINCT CASE WHEN array_contains(le.topics, 'sanction.control')
                           THEN o.asset_id END)                    AS flagged_by_them
FROM pep_dev.silver.ownership AS o
JOIN pep_dev.eval.reached AS r          ON r.node_id = o.owner_id      -- owner is listed or connected
JOIN pep_dev.silver.legal_entity AS le  ON le.canonical_id = o.asset_id
WHERE o.is_current AND NOT o.is_self_owning
  AND array_contains(o.datasets, 'c4ads_xinjiang_mining')
  AND NOT le.is_sanctioned

### The 71 they flag but we can't reach today
Test: are they reachable once ENDED links are included (`reached_all`)? If yes, their only connection to a listed party runs through ownership that has ended.

In [0]:
-- expect: 67 | 4
SELECT count_if(ra.node_id IS NOT NULL) AS reachable_only_via_ended_links,
       count_if(ra.node_id IS NULL)     AS not_reachable_at_all
FROM pep_dev.silver.legal_entity AS le
LEFT JOIN (SELECT node_id FROM pep_dev.eval.reached     WHERE round > 0) AS r  ON r.node_id  = le.canonical_id
LEFT JOIN (SELECT node_id FROM pep_dev.eval.reached_all WHERE round > 0) AS ra ON ra.node_id = le.canonical_id
WHERE array_contains(le.topics, 'sanction.control')   -- they flag it
  AND NOT le.is_sanctioned                            -- not listed itself
  AND r.node_id IS NULL                               -- we can't reach it today → the 71

### Where did those chains break? (walk up)
For 66 of the 67, the company's own owner link is still current, so the break is higher up. Walk up through current links among companies that lost their connection, then take the ENDED link from an owner that's still connected (or listed): the "break link".
Results are saved as tables (`eval.stale_67`, `eval.upstream`) so they survive restarts.

In [0]:
%python
# 1. The 67: flagged by them, not reachable today, reachable once ended links are included
stale_df = spark.sql("""
    SELECT le.canonical_id AS id
    FROM pep_dev.silver.legal_entity AS le
    LEFT JOIN (SELECT node_id FROM pep_dev.eval.reached     WHERE round > 0) AS r  ON r.node_id  = le.canonical_id
    JOIN      (SELECT node_id FROM pep_dev.eval.reached_all WHERE round > 0) AS ra ON ra.node_id = le.canonical_id
    WHERE array_contains(le.topics, 'sanction.control')
      AND NOT le.is_sanctioned
      AND r.node_id IS NULL
""")
stale_df.write.mode("overwrite").saveAsTable("pep_dev.eval.stale_67")
stale = {row.id for row in stale_df.collect()}
print("stale companies:", len(stale))                       # expect 67

# 2. Companies connected once but not today (connected ever MINUS connected today)
spark.sql("""
    CREATE OR REPLACE TEMP VIEW disconnected AS
    SELECT node_id FROM pep_dev.eval.reached_all WHERE round > 0
    EXCEPT
    SELECT node_id FROM pep_dev.eval.reached
""")

# 3. Walk UP from the 67 through current links, staying among disconnected companies
upstream, frontier = set(stale), set(stale)                 # everything found so far / newest layer
while frontier:                                             # stop when a layer adds nothing
    spark.createDataFrame([(x,) for x in frontier], ["id"]).createOrReplaceTempView("frontier")
    owners = {row.owner_id for row in spark.sql("""
        SELECT e.owner_id
        FROM pep_dev.eval.edges_current AS e
        JOIN frontier     AS f ON f.id = e.asset_id        -- links INTO the current layer
        JOIN disconnected AS d ON d.node_id = e.owner_id   -- whose owner also lost its connection
    """).collect()}
    frontier = owners - upstream                            # owners not seen before
    upstream |= frontier                                    # add them

spark.createDataFrame([(x,) for x in upstream], ["id"]) \
     .write.mode("overwrite").saveAsTable("pep_dev.eval.upstream")
print("companies in the disconnected chains:", len(upstream))

In [0]:
-- expect: 2022: 1 | 2023: 2 | 2024: 1 | 2025: 1 | 2026: 11
SELECT year(o.end_date) AS end_year, count(*) AS break_links
FROM pep_dev.silver.ownership AS o
JOIN pep_dev.eval.upstream AS u ON u.id = o.asset_id        -- into the disconnected chains
JOIN pep_dev.eval.reached  AS r ON r.node_id = o.owner_id   -- from an owner still connected (or listed)
WHERE NOT o.is_current AND NOT o.is_self_owning             -- and the link has ended
GROUP BY year(o.end_date)
ORDER BY end_year

In [0]:
-- Private: real people and companies. Never post these names publicly.
SELECT o.owner_id,
       coalesce(le_o.name, p.primary_name) AS owner_name,   -- company name, or person's name
       o.asset_id,
       le_a.name                           AS asset_name,
       o.end_date, o.pct_min, o.datasets
FROM pep_dev.silver.ownership AS o
JOIN pep_dev.eval.upstream AS u ON u.id = o.asset_id
JOIN pep_dev.eval.reached  AS r ON r.node_id = o.owner_id
LEFT JOIN pep_dev.silver.legal_entity AS le_o ON le_o.canonical_id = o.owner_id
LEFT JOIN pep_dev.silver.person       AS p    ON p.person_id = o.owner_id
LEFT JOIN pep_dev.silver.legal_entity AS le_a ON le_a.canonical_id = o.asset_id
WHERE NOT o.is_current AND NOT o.is_self_owning
ORDER BY o.end_date

### How many flagged companies sit below old vs recent breaks? (walk down)
Finding: 45 of the 67 sit below breaks from 2022-2025 (likely out of date), 22 below July 2026 breaks (plausible lag), none below both. Their own article says "An end date on a shareholding stops the chain".

In [0]:
%python
from datetime import date

stale = {row.id for row in spark.table("pep_dev.eval.stale_67").collect()}   # the 67

def flagged_below(break_assets):
    """Which of the 67 sit below these break points (following current links down)?"""
    below, frontier = set(break_assets), set(break_assets)
    while frontier:
        spark.createDataFrame([(x,) for x in frontier], ["id"]).createOrReplaceTempView("frontier")
        kids = {row.asset_id for row in spark.sql("""
            SELECT e.asset_id
            FROM pep_dev.eval.edges_current AS e
            JOIN frontier              AS f ON f.id = e.owner_id   -- links OUT of the current layer
            JOIN pep_dev.eval.upstream AS u ON u.id = e.asset_id   -- stay inside the disconnected chains
        """).collect()}
        frontier = kids - below                                     # new companies one level down
        below |= frontier
    return below & stale                                            # keep only the 67

breaks = spark.sql("""
    SELECT o.asset_id, o.end_date
    FROM pep_dev.silver.ownership AS o
    JOIN pep_dev.eval.upstream AS u ON u.id = o.asset_id
    JOIN pep_dev.eval.reached  AS r ON r.node_id = o.owner_id
    WHERE NOT o.is_current AND NOT o.is_self_owning
""").collect()

old    = flagged_below({b.asset_id for b in breaks if b.end_date <  date(2026, 1, 1)})
recent = flagged_below({b.asset_id for b in breaks if b.end_date >= date(2026, 1, 1)})
print("below 2022-2025 breaks:", len(old))                 # expect 45
print("below July 2026 breaks:", len(recent))              # expect 22
print("below both:            ", len(old & recent))        # expect 0

### Diagnostic: do the break points themselves still carry the flag?
- Break point unflagged, companies below flagged → the flag isn't retracted further down the chain.
- Break point flagged too → end dates aren't applied to these links.

In [0]:
SELECT o.asset_id, o.end_date,
       array_contains(le.topics, 'sanction.control') AS break_point_still_flagged
FROM pep_dev.silver.ownership AS o
JOIN pep_dev.eval.upstream AS u ON u.id = o.asset_id
JOIN pep_dev.eval.reached  AS r ON r.node_id = o.owner_id
JOIN pep_dev.silver.legal_entity AS le ON le.canonical_id = o.asset_id
WHERE NOT o.is_current AND NOT o.is_self_owning
ORDER BY o.end_date